# Задача 1. Среда GridWorld

Лабиринт `h x w`, окружённый стеной. Агент видит только цвет клетки, на которой стоит: one-hot вектор длины
`n_colors + 3` (цвета пола, затем цель, препятствие и стена). Действия: 0 вверх, 1 вправо, 2 вниз, 3 влево.
Награда 1 за попадание в цель, тогда эпизод заканчивается (`terminated`), иначе 0. Лимит `4·h·w` шагов (`truncated`).

Код: `src/envs/gridworld.py` и `src/envs/setups.py`. Результаты сохраняются в `outputs/task1/`.

In [ ]:
import csv
import os
import sys
from pathlib import Path

# работаем из корня репозитория
if Path.cwd().name == "tasks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np

from src import utils
from src.envs import GridWorldEnv, bfs_distances, make_env, optimal_stats
from src.envs.gridworld import DOWN, LEFT, RIGHT, UP

OUT = Path("outputs/task1")

In [ ]:
SETUPS = {
    1: dict(h=5, w=5, n_colors=None, obstacle_fraction=0.0),  # 25 уникальных цветов — полностью наблюдаемая
    2: dict(h=5, w=5, n_colors=5, obstacle_fraction=0.0),
    3: dict(h=10, w=10, n_colors=7, obstacle_fraction=0.1),
    4: dict(h=10, w=10, n_colors=4, obstacle_fraction=0.1),
}

In [ ]:
CHECKS = []  # результаты проверок, в конце сохраним в outputs/task1/validation/checks.csv


def check(name, ok):
    CHECKS.append({"check": name, "ok": bool(ok)})
    print("OK  " if ok else "ОШИБКА", name)

## Карты четырёх сред

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for i in SETUPS:
    env = make_env(**SETUPS[i], seed=0)
    env.reset(seed=0)
    utils.draw_env(axes[i - 1], env)
    axes[i - 1].set_title(f"Env{i}: {env.h}x{env.w}, цветов пола {env.n_colors}, препятствий {env.obstacle_mask.sum()}")
(OUT / "figures").mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "figures" / "setups.png", dpi=150, bbox_inches="tight")
plt.show()

## Характеристики сред

Oracle знает свои координаты и идёт кратчайшим путём (BFS) — это ориентир сверху, а не честный бейзлайн: у агентов
в задаче 2 координат нет. Случайная политика — ориентир снизу.

In [ ]:
def random_policy(env, n_episodes=200):
    # случайные действия: доля успехов, средняя длина и средняя дисконтированная отдача
    rng = np.random.default_rng(0)
    successes, lengths, returns = [], [], []
    for k in range(n_episodes):
        env.reset(seed=k)
        terminated, truncated = False, False
        while not (terminated or truncated):
            obs, reward, terminated, truncated, info = env.step(int(rng.integers(4)))
        successes.append(terminated)
        lengths.append(env.steps)
        returns.append(0.99 ** (env.steps - 1) * terminated)
    return np.mean(successes), np.mean(lengths), np.mean(returns)

In [ ]:
rows = []
for i in SETUPS:
    env = make_env(**SETUPS[i], seed=0)
    free = ~env.obstacle_mask
    dist = bfs_distances(free, env.pos_goal)
    check(f"Env{i}: все свободные клетки достижимы", (dist[free] >= 0).all())
    oracle = optimal_stats(env)
    rand_success, rand_length, rand_return = random_policy(env)
    rows.append({"env": f"Env{i}", "size": f"{env.h}x{env.w}", "floor_colors": env.n_colors,
                 "obstacles": int(env.obstacle_mask.sum()), "free_cells": int(free.sum()),
                 "cells_per_color": round(free.sum() / len(np.unique(env.floor_colors[free])), 2),
                 "max_steps": env.max_steps, "longest_path": int(dist.max()),
                 "oracle_length": round(oracle["length"], 2), "oracle_return": round(oracle["return"], 3),
                 "random_success": round(rand_success, 3), "random_length": round(rand_length, 1),
                 "random_return": round(rand_return, 3)})
    print(rows[-1])

In [ ]:
(OUT / "tables").mkdir(parents=True, exist_ok=True)
with open(OUT / "tables" / "env_summary.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

In [ ]:
# длина кратчайшего пути до цели из каждой клетки
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for i in SETUPS:
    env = make_env(**SETUPS[i], seed=0)
    dist = bfs_distances(~env.obstacle_mask, env.pos_goal).astype(float)
    dist[env.obstacle_mask] = np.nan
    image = axes[i - 1].imshow(dist, cmap="viridis")
    axes[i - 1].set_title(f"Env{i}: шагов до цели по кратчайшему пути")
    fig.colorbar(image, ax=axes[i - 1], fraction=0.046)
fig.savefig(OUT / "figures" / "bfs_distances.png", dpi=150, bbox_inches="tight")
plt.show()

## Правила среды на маленькой карте 3x3

In [ ]:
floor = np.array([[0, 0, 0], [1, 1, 1], [2, 2, 2]])
obstacles = np.zeros((3, 3), dtype=bool)
obstacles[1, 1] = True
env = GridWorldEnv(floor, (2, 2), n_colors=3, obstacle_mask=obstacles)
print(env.render())

In [ ]:
obs, info = env.reset(seed=0, options={"pos": (0, 0)})
obs, reward, terminated, truncated, info = env.step(UP)
check("удар о стену: агент на месте", tuple(info["pos"]) == (0, 0) and info["bumped"])
check("see_obstacle=True: видит цвет стены", np.argmax(obs) == env.wall_color)

env.reset(seed=0, options={"pos": (0, 1)})
obs, reward, terminated, truncated, info = env.step(DOWN)
check("удар о препятствие: агент на месте, видит препятствие", tuple(info["pos"]) == (0, 1) and np.argmax(obs) == env.obstacle_color)

env.reset(seed=0, options={"pos": (2, 1)})
obs, reward, terminated, truncated, info = env.step(RIGHT)
check("вход в цель: награда 1 и terminated", reward == 1.0 and terminated)

In [ ]:
# see_obstacle=False: при ударе агент видит цвет своей клетки
env_blind = GridWorldEnv(floor, (2, 2), n_colors=3, obstacle_mask=obstacles, see_obstacle=False)
env_blind.reset(seed=0, options={"pos": (0, 0)})
obs, reward, terminated, truncated, info = env_blind.step(LEFT)
check("see_obstacle=False: видит свою клетку", np.argmax(obs) == 0)

# лимит шагов
env_short = GridWorldEnv(floor, (2, 2), n_colors=3, obstacle_mask=obstacles, max_steps=3)
env_short.reset(seed=0, options={"pos": (0, 0)})
flags = [env_short.step(UP)[3] for _ in range(3)]
check("truncated ровно на 3-м шаге", flags == [False, False, True])

## Стартовые позиции

Старт не может быть на цели, препятствии или вне карты: в этом случае `ValueError`.

In [ ]:
for bad in [(2, 2), (1, 1), (-1, 0), (3, 0)]:
    try:
        GridWorldEnv(floor, (2, 2), n_colors=3, obstacle_mask=obstacles, pos_agent=bad)
        check(f"старт {bad} запрещён", False)
    except ValueError as error:
        check(f"старт {bad} запрещён: {error}", True)

In [ ]:
starts = set()
for s in range(500):
    obs, info = env.reset(seed=s)
    starts.add(tuple(info["pos"]))
check("случайный старт только со свободных клеток и не с цели", (1, 1) not in starts and (2, 2) not in starts)
check("встречаются все 7 допустимых клеток", len(starts) == 7)

# веса стартов: у (0, 0) вес 10, у остальных 6 клеток по 1
weights = np.ones((3, 3))
weights[0, 0] = 10
env_w = GridWorldEnv(floor, (2, 2), n_colors=3, obstacle_mask=obstacles, pos_agent=weights)
share = np.mean([tuple(env_w.reset(seed=s)[1]["pos"]) == (0, 0) for s in range(2000)])
check(f"доля старта из (0, 0) = {share:.2f}, ожидается {10 / 16:.2f}", abs(share - 10 / 16) < 0.05)

## Цветовое и координатное наблюдения

Координатный режим `obs_mode="coords"` нужен только для отладки: в нём задача становится полностью наблюдаемой.

In [ ]:
env_color = make_env(**SETUPS[2], seed=0)
env_coords = make_env(**SETUPS[2], seed=0, obs_mode="coords")
obs_color, info = env_color.reset(seed=1)
obs_coords, _ = env_coords.reset(seed=1)
print("позиция", tuple(info["pos"]), "цвет клетки", info["color"])
print("цветовое наблюдение:", obs_color)
print("координатное наблюдение: единица на месте", np.argmax(obs_coords))
check("цветовое наблюдение — one-hot цвета", obs_color.sum() == 1 and np.argmax(obs_color) == info["color"])
check("координатное наблюдение — one-hot позиции", np.argmax(obs_coords) == info["pos"][0] * 5 + info["pos"][1])

## Воспроизводимость

In [ ]:
a = make_env(**SETUPS[4], seed=0)
b = make_env(**SETUPS[4], seed=0)
c = make_env(**SETUPS[4], seed=1)
check("одинаковый seed — одинаковая карта и цель", np.array_equal(a.color_map, b.color_map) and a.pos_goal == b.pos_goal)
check("разный seed — разные карты", not np.array_equal(a.color_map, c.color_map))
starts_a = [tuple(a.reset(seed=s)[1]["pos"]) for s in range(20)]
starts_b = [tuple(b.reset(seed=s)[1]["pos"]) for s in range(20)]
check("одинаковый seed в reset — одинаковые старты", starts_a == starts_b)

In [ ]:
(OUT / "validation").mkdir(parents=True, exist_ok=True)
with open(OUT / "validation" / "checks.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["check", "ok"])
    writer.writeheader()
    writer.writerows(CHECKS)
print("проверок:", len(CHECKS), "| с ошибкой:", sum(not c["ok"] for c in CHECKS))